# Instacart Market Basket Analysis

## Milestone 1: Data Loading and Schema Understanding

This notebook performs the initial data ingestion and structural audit of the Instacart Online Grocery Shopping dataset (6 relational tables: orders, order_products__prior, order_products__train, products, aisles, departments), ahead of exploratory data analysis (EDA) and Market Basket Analysis (MBA).

### Objectives
- Load all Instacart source tables
- Inspect row/column structure
- Understand how the tables relate to each other (shared keys: `order_id`, `product_id`, `aisle_id`, `department_id`)
- Identify the grain of each table
- Validate data types, missing values, duplicates, and categorical distributions
- Prepare a clean, well-understood foundation for EDA and Market Basket Analysis

## 1.1 Import Libraries and Load Datasets

### Objective
Import the required libraries, configure display settings for readability, and load all six raw Instacart CSV files into memory. Datasets are stored in a dictionary (`datasets`) so that later steps can iterate over every table without repeating code.

In [27]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
#Display settings for better visualization
# ==========================================
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.2f}'.format)
print("Libraries imported successfully.")

# ==========================================
# File paths to the raw datasets
# ==========================================
orders_path = "../data/raw/orders.csv"
prior_path = "../data/raw/order_products__prior.csv"
train_path = "../data/raw/order_products__train.csv"
products_path = "../data/raw/products.csv"
aisles_path = "../data/raw/aisles.csv"
departments_path = "../data/raw/departments.csv"

# ==========================================
#Load all sources tables
# ==========================================
orders = pd.read_csv(orders_path)
order_products_prior = pd.read_csv(prior_path)
order_products_train = pd.read_csv(train_path)
products = pd.read_csv(products_path)
aisles = pd.read_csv(aisles_path)
departments = pd.read_csv(departments_path)

#==================================================================
# Store datasets in a dictionary for iteration across the notebook
#==================================================================
datasets = {
    "Orders": orders,
    "Order Products Prior": order_products_prior,
    "Order Products Train": order_products_train,
    "Products": products,
    "Aisles": aisles,
    "Departments": departments
}
print("All Datasets loaded successfully.")


Libraries imported successfully.
All Datasets loaded successfully.


## 1.2 Preview of Each Dataset

### Objective
Preview the first few rows of each dataset to sanity-check the data visually before running deeper checks.

In [28]:
#==========================================
#Preview top rows of each dataset
#==========================================

for name, df in datasets.items():
    print(f"preview of {name}")
    display(df.head())

preview of Orders


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


preview of Order Products Prior


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


preview of Order Products Train


...

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


preview of Aisles


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


preview of Departments


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


### Observation
- Row-level content matches the expected schema for each table (e.g. `orders` shows one row per order with sequencing fields like `order_number` and `days_since_prior_order`).
- `eval_set` in `orders` confirms the prior/train/test split used by Instacart's original competition design.

## 1.3 Dataset Shapes

### Objective
Check the number of rows and columns in each dataset to understand its scale and grain before further inspection.

In [29]:
#==========================================
#shape of the each datasets
#==========================================
for name, df in datasets.items():
    print(f"{name} shape: {df.shape}")

Orders shape: (3421083, 7)
Order Products Prior shape: (32434489, 4)
Order Products Train shape: (1384617, 4)
Products shape: (49688, 4)
Aisles shape: (134, 2)
Departments shape: (21, 2)


### Observation
- `order_products_prior` (32.4M rows) is by far the largest table, representing the full history of prior purchases, while `order_products_train` (1.4M rows) is a much smaller labeled subset held out for prediction tasks.
- `orders` (3.42M rows) sits at the order grain, one row per order.
- `products`, `aisles`, and `departments` are small lookup/dimension tables (49,688 / 134 / 21 rows respectively), each keyed by their own ID.

## 1.4 Dataset Columns

### Objective
Inspect the column structure of each dataset to identify the join keys that link the tables together and confirm the grain of each one.

In [30]:
# ==========================================
#Columns in each dataset
# ==========================================

for name, df in datasets.items():
    print(f"Columns in {name}: {list(df.columns)}\n")


Columns in Orders: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Columns in Order Products Prior: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Columns in Order Products Train: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Columns in Products: ['product_id', 'product_name', 'aisle_id', 'department_id']

Columns in Aisles: ['aisle_id', 'aisle']

Columns in Departments: ['department_id', 'department']



### Observation
- `order_id` links `orders` to `order_products_prior` / `order_products_train`.
- `product_id` links the order-product tables to `products`.
- `aisle_id` / `department_id` appear in `products`, `aisles`, `departments` → link products to their category hierarchy.
- This confirms a classic star-schema layout: `orders` and `order_products_*` are fact tables, while `products`, `aisles`, and `departments` are dimension tables.

## 1.5 Inspect Data Types

### Objective
Inspect the data type of each column across all datasets to verify that every variable is stored in the correct format before performing analysis.

In [31]:
# ==========================================
#Inspect the data types of each dataset
# ==========================================
for name, df in datasets.items():
    print(f"{name} data types:")
    print(df.dtypes,"\n")

Orders data types:
order_id                    int64
user_id                     int64
eval_set                      str
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object 

Order Products Prior data types:
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object 

Order Products Train data types:
order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object 

Products data types:
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object 

Aisles data types:
aisle_id    int64
aisle         str
dtype: object 

Departments data types:
department_id    int64
department         str
dtype: object 



### Observation
- Most columns contain whole numerical values such as identifiers and order-related information.
- The `days_since_prior_order` column may contain decimal values or missing (`NaN`) values.
- Text-based columns include `eval_set`, `product_name`, `aisle`, and `department`.
- No obvious data type inconsistencies were observed at this stage.

## 1.6 Grain, Primary Keys & Foreign Keys

### Objective

Identify the grain (level of detail), primary keys (PK), and foreign keys (FK) of each dataset to understand how the tables are related before performing joins.

| Dataset              | Grain (One Row = ...)              | Primary Key (PK)                            | Foreign Key (FK)            |
| -------------------- | ---------------------------------- | ------------------------------------------- | --------------------------- |
| Orders               | One order                          | `order_id`                                  | `user_id`                   |
| Order Products Prior | One product within one prior order | (`order_id`, `product_id`) *(Composite PK)* | `order_id`, `product_id`    |
| Order Products Train | One product within one train order | (`order_id`, `product_id`) *(Composite PK)* | `order_id`, `product_id`    |
| Products             | One product                        | `product_id`                                | `aisle_id`, `department_id` |
| Aisles               | One aisle                          | `aisle_id`                                  | —                           |
| Departments          | One department                     | `department_id`                             | —                           |

## 1.7 Missing Values Analysis

### Objective

Identify missing values across all datasets to evaluate data quality and determine whether any missing values require cleaning or represent meaningful business information.

In [32]:
#==========================================
# Missing Values Analysis
#==========================================

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Orders
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

Order Products Prior
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

Order Products Train
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

Products
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

Aisles
aisle_id    0
aisle       0
dtype: int64

Departments
department_id    0
department       0
dtype: int64


### Observation

- Only the `days_since_prior_order` column in the **Orders** dataset contains missing values (206,209 records, approximately **6.03%**).
- All other columns across the six datasets contain no missing values.
- The missing values appear to be limited to a single business-related attribute rather than indicating widespread data quality issues.

## 1.8 Duplicate Analysis

### Objective

Identify duplicate records across all datasets to ensure data integrity before performing analysis.

In [33]:
#==========================================
# Duplicate Analysis
#==========================================

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = df.duplicated().mean() * 100


    print(f"\n{name}")
    print(f"Duplicate Rows: {duplicate_count}")
    print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")


Orders
Duplicate Rows: 0
Duplicate Percentage: 0.00%

Order Products Prior
Duplicate Rows: 0
Duplicate Percentage: 0.00%

Order Products Train
Duplicate Rows: 0
Duplicate Percentage: 0.00%

Products
Duplicate Rows: 0
Duplicate Percentage: 0.00%

Aisles
Duplicate Rows: 0
Duplicate Percentage: 0.00%

Departments
Duplicate Rows: 0
Duplicate Percentage: 0.00%


### Observation

- No duplicate rows were found in any of the six datasets.
- This indicates that there are no exact duplicate records at the row level.
- Further validation is still required to check for missing values, invalid values, and relationship consistency between tables.

## 1.9 Categorical Value Validation

### Objective

Explore categorical columns to understand the distribution of categories and validate expected values.

In [34]:
#==========================================
# Categorical Value Validation
#==========================================

categorical_columns = {
    "Orders - eval_set": orders["eval_set"],
    "Order Products Prior - reordered": order_products_prior["reordered"],
    "Order Products Train - reordered": order_products_train["reordered"]
}

for name, column in categorical_columns.items():
    print("=" * 50)
    print(name)
    print("=" * 50)
    print(column.value_counts())
    print()

Orders - eval_set
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

Order Products Prior - reordered
reordered
1    19126536
0    13307953
Name: count, dtype: int64

Order Products Train - reordered
reordered
1    828824
0    555793
Name: count, dtype: int64



### Observation
- The `eval_set` column contains the expected categories: `prior`, `train`, and `test`.
- The `prior` dataset contains the majority of orders, indicating that most records represent customers' historical purchases.
- The `reordered` column contains only valid binary values (`0` and `1`) in both prior and train datasets.
- Reordered products (`1`) occur more frequently than non-reordered products (`0`), suggesting that customers often repurchase previously bought items.

## 1.10 Dataset Information

### Objective

Inspect the structure of each dataset, including the number of rows, columns, data types, non-null values, and memory usage.

In [35]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    df.info()
    print()

Orders
<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB

Order Products Prior
<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
dtypes: int64(4)
memory usage: 989.8 MB

Order Products Train
<class 'pandas.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 4 columns):
 #   Column             Non-Nul


### Observation

- Dataset structures were successfully verified.
- Most columns have appropriate data types (`int64` and `object`/`str`).
- Only `days_since_prior_order` contains missing values, while all other columns are complete.
- Memory usage is reasonable for the size of each dataset.

## 1.11 Statistical Summary

### Objective

Generate summary statistics for numerical columns to understand their distribution, central tendency, and variability.

In [36]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    display(df.describe())
    print()

Orders


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,"3,421,083.00","3,421,083.00","3,421,083.00","3,421,083.00","3,421,083.00","3,214,874.00"
mean,"1,710,542.00","102,978.21",17.15,2.78,13.45,11.11
std,"987,581.74","59,533.72",17.73,2.05,4.23,9.21
min,1.00,1.00,1.00,0.00,0.00,0.00
25%,"855,271.50","51,394.00",5.00,1.00,10.00,4.00
50%,"1,710,542.00","102,689.00",11.00,3.00,13.00,7.00
75%,"2,565,812.50","154,385.00",23.00,5.00,16.00,15.00
max,"3,421,083.00","206,209.00",100.00,6.00,23.00,30.00



Order Products Prior


,order_id,product_id,add_to_cart_order,reordered
count,"32,434,489.00","32,434,489.00","32,434,489.00","32,434,489.00"
mean,"1,710,748.52","25,576.34",8.35,0.59
std,"987,300.70","14,096.69",7.13,0.49
min,2.00,1.00,1.00,0.00
25%,"855,943.00","13,530.00",3.00,0.00
50%,"1,711,048.00","25,256.00",6.00,1.00
75%,"2,565,514.00","37,935.00",11.00,1.00
max,"3,421,083.00","49,688.00",145.00,1.00



Order Products Train


...

,product_id,aisle_id,department_id
count,"49,688.00","49,688.00","49,688.00"
mean,"24,844.50",67.77,11.73
std,"14,343.83",38.32,5.85
min,1.00,1.00,1.00
25%,"12,422.75",35.00,7.00
50%,"24,844.50",69.00,13.00
75%,"37,266.25",100.00,17.00
max,"49,688.00",134.00,21.00



Aisles


,aisle_id
count,134.00
mean,67.50
std,38.83
min,1.00
25%,34.25
50%,67.50
75%,100.75
max,134.00



Departments


,department_id
count,21.00
mean,11.00
std,6.20
min,1.00
25%,6.00
50%,11.00
75%,16.00
max,21.00


### Observation

- The statistical summary confirms that all numerical columns contain valid values.
- Customers placed an average of approximately 17 orders.
- Orders are typically placed around 1 PM (`order_hour_of_day` mean ≈ 13.45).
- Customers reorder after approximately 11 days on average.
- The dataset contains customers with up to 100 orders, indicating varying purchasing behavior.

### Next Steps
With schema, types, missing values, duplicates, and category distributions validated, the notebook is ready to proceed to exploratory data analysis (EDA) and feature engineering for Market Basket Analysis.

# Milestone 1 Summary

### Key Findings

- Successfully loaded and validated all six datasets.
- Identified dataset grain, primary keys, and foreign keys.
- Verified that only `days_since_prior_order` contains missing values.
- No duplicate rows were found.
- Validated categorical columns and confirmed expected values.
- Generated statistical summaries to understand the overall data distribution.

### Conclusion

The datasets are well-structured and ready for data cleaning, feature engineering, and business analysis in the next milestone.